# New version, using the .parquet exports
instead of synchronizing the complete postgresql database all the time, i can export PARAM and METRIC .parquet files per group_id and only synchronize/copy what i need.

In [10]:
import sys
import os
project_root = os.path.abspath('..')          # one level up from notebooks' directory
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
import numpy as np
from pathlib import Path
from key_mapping import key_mapping
from data_preparation import add_grouped_mean_columns, get_multi_carrier_metrics

In [ ]:
# pandas version
def make_single_step_multi_carrier_plot_data(params_file:str):
    from key_mapping import key_mapping
    from data_preparation import add_grouped_mean_columns, get_multi_carrier_metrics
    print(f'Reading params file {params_file}')
    PARAMS = pd.read_parquet(f"../data/experiments/exports/{params_file}", )
    print('Reading metrics file')
    METRICS = pd.read_parquet(f"../data/experiments/exports/{params_file.replace('params', 'metrics')}", )

    print('Pivot METRICS')
    METRICS_WIDE = METRICS.pivot(columns='key', index=['run_uuid', 'step'], values='value').reset_index()
    print('Get single step metrics')
    single_step_metrics = [col for col in METRICS_WIDE.columns if pd.isna(METRICS_WIDE[METRICS_WIDE['step'] > 0][col]).all()]
    SS_METRICS = METRICS_WIDE[METRICS_WIDE['step'] == 0][['run_uuid'] + single_step_metrics]
    SS_MERGED = pd.merge(PARAMS, SS_METRICS, on='run_uuid')
    print('Add grouped columns')
    add_grouped_mean_columns(SS_MERGED, inplace=True)

    # single step, multi-carrier metrics
    print('Get multi-carrier metrcis')
    SS_MC_METRICS = get_multi_carrier_metrics(SS_METRICS)
    SS_MC_MERGED = pd.merge(PARAMS, SS_MC_METRICS, on='run_uuid')

    # Multi-step metrics
    # print('Get multi-step metrics')
    # multi_step_metrics = [col for col in METRICS_WIDE.columns if col not in single_step_metrics]
    # MS_METRICS = METRICS_WIDE[multi_step_metrics]
    # MS_METRICS.update(MS_METRICS.groupby('run_uuid').ffill())  # forward fill the multi-step metrics that might have gaps
    # MS_METRICS = MS_METRICS.replace(np.inf, np.nan)
    # add_grouped_mean_columns(MS_METRICS, inplace=True)
    # MS_MERGED = pd.merge(PARAMS, MS_METRICS, on='run_uuid')

    # to wide format
    print('Pivot SS_MC_MERGED to wide format')
    SS_MC_MERGED_WIDE:pd.DataFrame = SS_MC_MERGED.pivot(columns='key', values='value', index=PARAMS.columns.to_list() + ['carrier']).reset_index()
    SS_MC_MERGED_WIDE = SS_MC_MERGED_WIDE.rename(columns=key_mapping)

    print('Write parquet file to disk')
    write_file =params_file.split('-')[0] + '-SSC_MC_MERGED_WIDE.parquet'
    write_file = f"../data/experiments/plot_ready/{write_file}"
    SS_MC_MERGED_WIDE.to_parquet(write_file)
    print(f'Done. File is at {write_file}')

In [19]:
# tags =["2026-03-31_12-34_hpc3_P3.1.10", "P3.1.14", "P3.1.15", "P3.1.16-v1", "P3.1.17-v2"]
# tags = ["P3.1.17"]
# tags = ["P3.1.16-new"]
# tags = ["P3.1.17-v2"]
for foo in ['14', '15', '16', '17', '19', '20', '21', '22']:
    params_file = f"['P3.1.{foo}']-params.parquet"
    make_single_step_multi_carrier_plot_data(params_file)

Reading params file
Reading metrics file
Pivot METRICS
Get single step metrics
Add grouped columns
Identified groups: ['my_MinWBMP', 'my_convex_hull_jaccard_distance', 'my_hausdorff_distance', 'my_modified_hausdorff_distance', 'my_tsp_hull_jaccard_distance', 'my_tsp_obj_val_diff', 'target_opt_final']
Get multi-carrier metrcis
Pivot SS_MC_MERGED to wide format
Write parquet file to disk
Done. File is at ../data/experiments/plot_ready/['P3.1.14']-SSC_MC_MERGED_WIDE.parquet
Reading params file
Reading metrics file
Pivot METRICS
Get single step metrics
Add grouped columns
Identified groups: ['my_MinWBMP', 'my_convex_hull_jaccard_distance', 'my_hausdorff_distance', 'my_modified_hausdorff_distance', 'my_tsp_hull_jaccard_distance', 'my_tsp_obj_val_diff', 'target_opt_final']
Get multi-carrier metrcis
Pivot SS_MC_MERGED to wide format
Write parquet file to disk
Done. File is at ../data/experiments/plot_ready/['P3.1.15']-SSC_MC_MERGED_WIDE.parquet
Reading params file
Reading metrics file
Pivot M

In [5]:
non_plot_params = [
    'run_uuid',
    'name_x',
    'source_type',
    'source_name',
    'entry_point_name',
    'user_id',
    'status',
    'start_time',
    'end_time',
    'source_version',
    'lifecycle_stage_x',
    'artifact_uri',
    'experiment_id',
    'deleted_time',
    'name_y',
    'artifact_location',
    'lifecycle_stage_y',
    'creation_time',
    'last_update_time',
    'group_id',
    'instance_id',
    'mlflow.runName',
    'mlflow.source.git.commit',
    'mlflow.source.name',
    'mlflow.source.type',
    'mlflow.user',
    'mlflow.loggedArtifacts',
]
plot_params = [x for x in PARAMS.columns if x not in non_plot_params ]
non_unique_params: pd.DataFrame = PARAMS.loc[:, PARAMS.nunique() != 1]
non_unique_params = non_unique_params.drop(columns=non_plot_params, errors='ignore', inplace=False)
print("List of non-unique parameters:")
list(non_unique_params.columns)
# sort non-unique first
plot_params = [None] + list(non_unique_params) + [x for x in plot_params if x not in non_unique_params]

# get all metrics
plot_metrics = (SS_MC_MERGED["key"].unique())

List of non-unique parameters:


## Plots

In [21]:
SS_MC_MERGED_WIDE = pd.read_parquet("../data/experiments/plot_ready/['P3.1.14']-SSC_MC_MERGED_WIDE.parquet")
SS_MC_MERGED_WIDE.head()

key,run_uuid,name_x,source_type,source_name,entry_point_name,user_id,status,start_time,end_time,source_version,...,TWL,solver__time_window_selection,carrier,$d_{MinWBMP}$,$d_{J}$,$d_{PHD}$,$d_{MPHD}$,$d_{THJD}$,$d_{TSP}$,target_opt_final
0,0001a813c8f94ee7942a3019fdc03e91,wise-shark-742,UNKNOWN,NaN,NaN,ag_em-elting,FINISHED,1776283015633,1776283570338,NaN,...,8.640000e+13,UniformPreference,0,108.110045,0.390318,25.999769,20.912395,0.390318,30.2472,5.196152
1,0001a813c8f94ee7942a3019fdc03e91,wise-shark-742,UNKNOWN,NaN,NaN,ag_em-elting,FINISHED,1776283015633,1776283570338,NaN,...,8.640000e+13,UniformPreference,1,63.823062,0.876562,34.168310,14.731542,0.838610,64.2810,18.417044
2,0001a813c8f94ee7942a3019fdc03e91,wise-shark-742,UNKNOWN,NaN,NaN,ag_em-elting,FINISHED,1776283015633,1776283570338,NaN,...,8.640000e+13,UniformPreference,2,95.657472,0.866861,34.153952,20.575544,0.878870,20.3403,10.371234
3,0017dedb49294c9494bb2c1d9bc20be9,marvelous-whale-223,UNKNOWN,NaN,NaN,ag_em-elting,FINISHED,1776265351381,1776265930853,NaN,...,8.640000e+13,UniformPreference,0,180.558677,0.582277,32.332105,26.963031,0.582277,98.2084,7.574629
4,0017dedb49294c9494bb2c1d9bc20be9,marvelous-whale-223,UNKNOWN,NaN,NaN,ag_em-elting,FINISHED,1776265351381,1776265930853,NaN,...,8.640000e+13,UniformPreference,1,161.352842,1.000000,58.950623,37.800209,1.000000,115.2619,13.006008


In [24]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotnine as p9
import os

# --- 1. Global holder for the current plot object ---
current_p = [None]

# --- 2. Define the plotting function logic ---
def create_plot(df, x, y, fill, linetype, cols, rows, hide_x_axis, scale_y_log10):
    # (Setting up labels and aesthetics - same as your logic)
    ylab = df["Error Function"][0] if y == "target_opt_final" else y
    
    aes_kwargs = {'x': x, 'y': y}
    if fill not in [None, "None"]: aes_kwargs['fill'] = fill
    if linetype not in [None, "None"]: aes_kwargs['linetype'] = linetype
    
    my_aes = p9.aes(**aes_kwargs)

    p = (
        p9.ggplot(df, my_aes)
        + p9.geom_boxplot()
        + my_p9_theme
        + p9.theme(figure_size=(12, 6))
        + my_scale_color_and_fill
        + p9.labs(y=ylab)
    )
    
    if scale_y_log10: p += p9.scale_y_log10()
    if hide_x_axis:
        p += p9.theme(axis_text_x=p9.element_blank(), 
                      axis_ticks_x=p9.element_blank(), 
                      axis_title_x=p9.element_blank())
    
    if cols or rows:
        p += p9.facet_grid(rows=rows, cols=cols)

    # Update the global reference for the save button
    current_p[0] = p
    
    # --- CRITICAL CHANGE START ---
    # In interactive_output, we must explicitly display the plot
    display(p) 
    
    # Print stats below the plot
    group_keys = [key for key in [fill, cols, rows, linetype] if key not in [None, "None"]]
    if group_keys:
        n_per_series = df.groupby(group_keys)[y].count()
        print(f"\nn per boxplot:\n{n_per_series}")
    # --- CRITICAL CHANGE END ---

# --- 3. Define Widgets (Keep these exactly as they were) ---
ui_x = widgets.Dropdown(options=plot_params, value=None, description='X Axis:')
ui_y = widgets.Dropdown(options=plot_metrics, value="target_opt_final", description='Y Axis:')
ui_fill = widgets.Dropdown(options=plot_params, value="Optimizer", description='Fill Color:')
ui_line = widgets.Dropdown(options=plot_params, value="Optimizer", description='Linetype:')
ui_cols = widgets.Dropdown(options=[None] + plot_params, value="# clusters", description='Facet Cols:')
ui_rows = widgets.Dropdown(options=[None] + plot_params, value="data__num_requests_per_carrier", description='Facet Rows:')
ui_hide_x = widgets.Checkbox(value=False, description="Hide x axis")
ui_log = widgets.Checkbox(value=False, description="Log y axis")

ui_filename = widgets.Text(value='my_plot.png', description='Filename:')
ui_save_btn = widgets.Button(description="💾 Save Figure", button_style='success')
ui_save_output = widgets.Output()

def on_save_clicked(b):
    with ui_save_output:
        clear_output()
        if current_p[0] is not None:
            current_p[0].save(ui_filename.value, verbose=False)
            print(f"✅ Saved!")
        else:
            print("❌ Error")

ui_save_btn.on_click(on_save_clicked)

# --- 4. Linking and Displaying ---
# This widget captures the output of 'create_plot'
out = widgets.interactive_output(
    create_plot, 
    {
        'df': widgets.fixed(SS_MC_MERGED_WIDE),
        'x': ui_x, 'y': ui_y, 'fill': ui_fill, 'linetype': ui_line,
        'cols': ui_cols, 'rows': ui_rows, 
        'hide_x_axis': ui_hide_x, 'scale_y_log10': ui_log
    }
)

controls = widgets.VBox([
    widgets.HBox([ui_x, ui_y, ui_fill]),
    widgets.HBox([ui_line, ui_cols, ui_rows]),
    widgets.HBox([ui_hide_x, ui_log]),
    widgets.HBox([ui_filename, ui_save_btn, ui_save_output])
])

# Use display() to show the UI and the output area
display(controls, out)

Output()